In [11]:
from pathlib import Path
import os
import pandas as pd
from sklearn.linear_model import LinearRegression
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

In [4]:
current_dir = Path.cwd()
while current_dir.name != "nyc-walkability-accident-analysis" and current_dir != current_dir.parent:
    current_dir = current_dir.parent

DATA_PATH = current_dir / "data" / "cleaned" / "final_data.csv"


df = pd.read_csv(DATA_PATH)

In [ ]:
def insight1_regression():
    X = df.dropna()[['total_population', 'Walk Score']]
    y = df.dropna()['NUMBER OF PEDESTRIANS AFFECTED']

    model = LinearRegression().fit(X, y)
    print(f"Walk Score and Population vs Pedestrians Affected R^2: {model.score(X, y)}")

def insight1_quantiles():
    top_10_walk_score = df[df['Walk Score'] >= df['Walk Score'].quantile(0.9)] 
    bottom_10_walk_score = df[df['Walk Score'] <= df['Walk Score'].quantile(0.1)] 
    
    top_pedestrian_risk = top_10_walk_score.groupby('zip_code')['NUMBER OF PEDESTRIANS AFFECTED'].sum().mean()
    bottom_pedestrian_risk = bottom_10_walk_score.groupby('zip_code')['NUMBER OF PEDESTRIANS AFFECTED'].sum().mean()
    
    print(f"Top 10% Risk: {top_pedestrian_risk}")
    print(f"Bottom 10% Risk: {bottom_pedestrian_risk}")
    print(f"Percent Difference: {((top_pedestrian_risk - bottom_pedestrian_risk) / bottom_pedestrian_risk) * 100}%")

############ Function Call ############
insight1_regression()
insight1_quantiles()

Walk Score and Population vs Pedestrians Affected R^2: 0.0009731198157182375
Top 10% Risk: 657.0
Bottom 10% Risk: 487.63157894736844
Percent Difference: 34.73286562331354%


In [6]:
def insight2_income_injuries():
  cluster_df = df.groupby('ZIP CODE').agg({
    'median_household_income': 'mean', 
    'NUMBER OF PEDESTRIANS AFFECTED': 'sum'
  })

  cluster_df = cluster_df.dropna()
  
  kmeans = KMeans(n_clusters=3, random_state=42)
  kmeans.fit(cluster_df)

  print("Income vs Injuries:")
  print(kmeans.cluster_centers_)

def insight2_income_walkscores():
  cluster_df = df.groupby('zip_code').agg({
    'median_household_income': 'mean',
    'Walk Score': 'mean'
  })

  cluster_df = cluster_df.dropna()

  kmeans = KMeans(n_clusters=3, random_state=42)
  kmeans.fit(cluster_df)

  print("Income vs Walk Score:")
  print(kmeans.cluster_centers_)

############ Function Call ############
insight2_income_injuries()
insight2_income_walkscores()

Income vs Injuries:
[[ 70908.33043478    953.03478261]
 [125888.86792453    466.83018868]
 [206533.625         302.        ]]
Income vs Walk Score:
[[5.36178444e+04 8.75000000e+01]
 [9.20186667e+04 8.64705882e+01]
 [1.46591381e+05 9.27142857e+01]]


In [ ]:
def insight3():
    df['Income Levels'] = pd.qcut(df['median_household_income'], q=3, labels=["Low", "Medium", "High"]) 
    pedestrians_by_income = df.groupby('Income Levels', observed=True)['NUMBER OF PEDESTRIANS AFFECTED'].mean()
    
    print("Average Pedestrians per Crash by Income Level:")
    print(pedestrians_by_income)

############ Function Call ############
insight3()

Average Pedestrians per Crash by Income Level:
Income Levels
Low       0.082430
Medium    0.071340
High      0.059713
Name: NUMBER OF PEDESTRIANS AFFECTED, dtype: float64


In [ ]:
def insight4():
  borough_risks = df.groupby('BOROUGH').apply(
    lambda x: (x['NUMBER OF PEDESTRIANS AFFECTED'].sum() / x['total_population'].sum()) * 10000,
    include_groups=False
  )

  print("Average Pedestrian Injuries and Deaths by Borough")
  print(borough_risks)

############ Function Call ############
insight4()

Average Pedestrian Injuries and Deaths by Borough
BOROUGH
BRONX            0.012503
BROOKLYN         0.010068
MANHATTAN        0.016848
QUEENS           0.012063
STATEN ISLAND    0.009837
dtype: float64


In [9]:
def insight5():
  df['Primary_Commute'] = df.apply(
    lambda x: 'Walk' if x['walk_to_work'] > x['public_transport_work'] else 'Transit', axis=1
  )

  commute_pedestrians  = df.groupby('Primary_Commute')['NUMBER OF PEDESTRIANS AFFECTED'].mean()
  
  print(commute_pedestrians)

############ Function Call ############
insight5()

Primary_Commute
Transit    0.071373
Walk       0.067189
Name: NUMBER OF PEDESTRIANS AFFECTED, dtype: float64


Data Visualizations

In [14]:
def visual1_pedestrians():
    visualization_df = df.groupby('zip_code').agg({
        'Walk Score': 'mean',
        'NUMBER OF PEDESTRIANS AFFECTED': 'sum'
    })

    visualization_df = visualization_df.dropna()
    
    x = visualization_df['Walk Score']
    y = visualization_df['NUMBER OF PEDESTRIANS AFFECTED']
    
    z = np.polyfit(x, y, 1)
    p = np.poly1d(z)

    plt.scatter(x, y)
    plt.plot(x, p(x), color='red', label="Trendline")
    
    plt.title("Walkability Score vs. Pedestrian Injuries and Deaths")
    plt.xlabel("Walk Score")
    plt.ylabel("Total Pedestrians Affected")
    
    plt.legend()
    plt.grid(True, linestyle='--')

    current_dir = Path(os.getcwd()).resolve()
    if current_dir.name in ["notebooks", "src"]:
        project_root = current_dir.parent
    else:
        project_root = current_dir
    CLEANED_FOLDER = project_root / "output" / "visuals"
    OUTPUT_FILE_PATH = CLEANED_FOLDER / "visual1_pedestrians"
    CLEANED_FOLDER.mkdir(parents=True, exist_ok=True)

    plt.savefig(OUTPUT_FILE_PATH)
    # plt.show()

def visual1_income():
    visualization_df = df.groupby('zip_code').agg({
        'median_household_income': 'mean',
        'Walk Score': 'mean'
    })

    visualization_df = visualization_df.dropna()
    
    x = visualization_df['median_household_income']
    y = visualization_df['Walk Score']
    
    z = np.polyfit(x, y, 1)
    p = np.poly1d(z)

    plt.scatter(x, y)
    plt.plot(x, p(x), color='red', label="Trendline")
    
    plt.title("Income vs. Walkability")
    plt.xlabel("Median Household Income")
    plt.ylabel("Walk Score")
    
    plt.legend()
    plt.grid(True, linestyle='--')

    current_dir = Path(os.getcwd()).resolve()
    if current_dir.name in ["notebooks", "src"]:
        project_root = current_dir.parent
    else:
        project_root = current_dir
    CLEANED_FOLDER = project_root / "output" / "visuals"
    OUTPUT_FILE_PATH = CLEANED_FOLDER / "visual1_income"
    CLEANED_FOLDER.mkdir(parents=True, exist_ok=True)

    plt.savefig(OUTPUT_FILE_PATH)
    # plt.show()
    plt.close()


############ Function Call ############
visual1_pedestrians()
visual1_income()

In [ ]:
def visual2_pedestrians():
    borough_risk = df.groupby('BOROUGH').apply(
        lambda x: (x['NUMBER OF PEDESTRIANS AFFECTED'].sum() / x['total_population'].sum()) * 10000,
        include_groups=False
    )

    plt.figure(figsize=(10,14))

    borough_risk.plot(kind='bar')

    plt.title("Pedestrian Risk per 10,000 Residents by Borough")
    plt.ylabel("Risk of Pedestrian Injury or Death per 10,000")
    plt.xlabel("Borough")

    current_dir = Path(os.getcwd()).resolve()
    if current_dir.name in ["notebooks", "src"]:
        project_root = current_dir.parent
    else:
        project_root = current_dir
    CLEANED_FOLDER = project_root / "output" / "visuals"
    OUTPUT_FILE_PATH = CLEANED_FOLDER / "visual2_pedestrians"
    CLEANED_FOLDER.mkdir(parents=True, exist_ok=True)

    plt.savefig(OUTPUT_FILE_PATH)

    # plt.show()
    plt.close()

def visual2_income():
    borough_income = df.groupby('BOROUGH')['median_household_income'].mean()
    
    fig, ax = plt.subplots(figsize=(10,14))
    
    borough_income.plot(kind='bar', ax=ax)
    
    plt.title("Average Median Household Income by Borough")
    plt.ylabel("Average Median Household Income ($)")
    plt.xlabel("Borough")

    current_dir = Path(os.getcwd()).resolve()
    if current_dir.name in ["notebooks", "src"]:
        project_root = current_dir.parent
    else:
        project_root = current_dir
    CLEANED_FOLDER = project_root / "output" / "visuals"
    OUTPUT_FILE_PATH = CLEANED_FOLDER / "visual2_income"
    CLEANED_FOLDER.mkdir(parents=True, exist_ok=True)

    plt.savefig(OUTPUT_FILE_PATH)
    
    # plt.show()
    plt.close()


############ Function Call ############
visual2_pedestrians()
visual2_income()

In [17]:
def visual3():
  yearly = df[df['CRASH YEAR'] != 2026].groupby('CRASH YEAR')['NUMBER OF PEDESTRIANS AFFECTED'].sum()

  plt.figure(figsize=(10,6))

  yearly.plot(kind = 'line', marker = 'o')
  plt.title("Total Pedestrian Injuries and Deaths")
  plt.xlabel("Year")
  plt.ylabel("Total Pedestrians Affected")

  plt.ylim(ymin=0)
  
  plt.grid(True, linestyle='--')

  current_dir = Path(os.getcwd()).resolve()
  if current_dir.name in ["notebooks", "src"]:
      project_root = current_dir.parent
  else:
      project_root = current_dir
  CLEANED_FOLDER = project_root / "output" / "visuals"
  OUTPUT_FILE_PATH = CLEANED_FOLDER / "visual3"
  CLEANED_FOLDER.mkdir(parents=True, exist_ok=True)

  plt.savefig(OUTPUT_FILE_PATH)

  # plt.show()
  plt.close()

############ Function Call ############
visual3()